In [1]:
import pandas as pd, numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
import statsmodels.formula.api as smf
import warnings; warnings.filterwarnings("ignore")

m = pd.read_csv("long.csv")
reg = m.dropna(subset=["debt_lag", "pb"]).sort_values("year").reset_index(drop=True)

def report(series, name):
    s = series.dropna()
    adf_p = adfuller(s, autolag="AIC")[1]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        kpss_p = kpss(s, regression="c", nlags="auto")[1]
    # ADF: low p = stationary (reject unit root). KPSS: low p = NON-stationary.
    print(f"{name:18s}  ADF p={adf_p:.3f}  KPSS p={kpss_p:.3f}")

print("STATIONARITY OF LEVELS  (ADF low=stationary; KPSS low=non-stationary)")
print("-"*60)
report(reg["pb"], "primary balance")
report(reg["debt"], "debt/GDP")
report(reg["debt"].diff(), "Δ debt/GDP")

# --- the spurious-regression concern & the standard fix ---
# If debt is I(1) and pb is I(0), the levels regression can be unbalanced.
# Two robustness responses:

print("\nROBUSTNESS TO PERSISTENCE")
print("-"*60)

# (1) Newey-West / HAC standard errors (corrects inference for serial correlation)
m_hac = smf.ols("pb ~ debt_lag", data=reg).fit(cov_type="HAC", cov_kwds={"maxlags":3})
print(f"Full sample, HAC SEs:   beta={m_hac.params['debt_lag']:+.4f}  p={m_hac.pvalues['debt_lag']:.3f}")

# (2) add lagged primary balance (dynamic spec — controls for persistence in pb)
reg["pb_lag"] = reg["pb"].shift(1)
m_dyn = smf.ols("pb ~ debt_lag + pb_lag", data=reg.dropna(subset=['pb_lag'])).fit()
print(f"With lagged pb (dynamic): beta={m_dyn.params['debt_lag']:+.4f}  p={m_dyn.pvalues['debt_lag']:.3f}")

# (3) first-differences spec (removes any unit root — very conservative)
reg["d_pb"] = reg["pb"].diff()
reg["d_debtlag"] = reg["debt_lag"].diff()
m_fd = smf.ols("d_pb ~ d_debtlag", data=reg.dropna(subset=['d_pb','d_debtlag'])).fit()
print(f"First differences:       beta={m_fd.params['d_debtlag']:+.4f}  p={m_fd.pvalues['d_debtlag']:.3f}")

# --- re-run the KEY era result (2000-16) with HAC, since that's the headline ---
print("\nHEADLINE ERA (2000-16) ROBUSTNESS")
print("-"*60)
s = reg.query("2000 <= year <= 2016")
m_era = smf.ols("pb ~ debt_lag", data=s).fit(cov_type="HAC", cov_kwds={"maxlags":2})
print(f"2000-16, HAC SEs:  beta={m_era.params['debt_lag']:+.4f}  p={m_era.pvalues['debt_lag']:.3f}")

STATIONARITY OF LEVELS  (ADF low=stationary; KPSS low=non-stationary)
------------------------------------------------------------
primary balance     ADF p=0.000  KPSS p=0.100
debt/GDP            ADF p=0.340  KPSS p=0.045
Δ debt/GDP          ADF p=0.000  KPSS p=0.100

ROBUSTNESS TO PERSISTENCE
------------------------------------------------------------
Full sample, HAC SEs:   beta=+0.0040  p=0.830
With lagged pb (dynamic): beta=+0.0097  p=0.070
First differences:       beta=+0.0065  p=0.807

HEADLINE ERA (2000-16) ROBUSTNESS
------------------------------------------------------------
2000-16, HAC SEs:  beta=-0.0322  p=0.000
